# TRIAGE PRO: End-to-End Clinical ML Pipeline

This notebook presents a full reproducible pipeline for early clinical risk prediction under epidemic conditions.

The goal is to construct a model that can estimate patient risk at the moment of hospital admission, using only information available at that stage (EHR-lite setting).

This design reflects real-world clinical constraints, where decisions must be made under uncertainty and with limited data.

The pipeline covers:
- raw data ingestion and harmonization;
- feature construction under clinical constraints;
- temporal validation across epidemic waves;
- predictive modeling and calibration;
- clinical utility assessment (decision curve analysis);
- interpretability and deployment-oriented demo.

Key principles:
- reproducibility;
- robustness to distribution shift;
- clinically meaningful modeling;
- interpretable risk estimation.

## 1. Project setup

This section defines the project structure and directory layout used throughout the pipeline. Separate folders are used for raw data, intermediate outputs, cleaned datasets, model-ready tables, figures, and summary tables.

In [ ]:
import os
import json

# Project directories (relative paths)
ROOT = "./data"

RAW = os.path.join(ROOT, "raw")
INTERIM = os.path.join(ROOT, "interim")
CLEAN = os.path.join(ROOT, "clean")
MR = os.path.join(ROOT, "modelready")
FIGS = os.path.join(ROOT, "figs")
TABLES = os.path.join(ROOT, "tables")

## 2. Source data and target feature groups

We specify the input files for each epidemic year and define the core variables used in the study. The selected feature groups include demographic variables, admission-time symptoms, and major comorbidities available in the original surveillance records.

In [ ]:
import os
from pathlib import Path
from urllib.request import urlretrieve

YEARS = [2020, 2021, 2022]

URLS = {
    2020: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2020/INFLUD20-26-06-2025.csv",
    2021: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2021/INFLUD21-26-06-2025.csv",
    2022: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2022/INFLUD22-26-06-2025.csv",
}

RAW_DIR = Path("./data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

FILES = {year: RAW_DIR / f"INFLUD{str(year)[-2:]}.csv" for year in YEARS}

for year, url in URLS.items():
    if not FILES[year].exists():
        urlretrieve(url, FILES[year])

In [ ]:
from pathlib import Path
import gc
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

DATA_DIR = Path("./data")
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"

In [ ]:
YEARS = [2020, 2021, 2022]

CSV_FILES = {
    2020: RAW_DIR / "INFLUD20.csv",
    2021: RAW_DIR / "INFLUD21.csv",
    2022: RAW_DIR / "INFLUD22.csv",
}

CORE_FEATURES = ["DT_SIN_PRI", "EVOLUCAO", "NU_IDADE_N", "CS_SEXO"]

SYMPTOM_FEATURES = [
    "FEBRE", "TOSSE", "GARGANTA", "DISPNEIA", "DESC_RESP", "SATURACAO",
    "DIARREIA", "VOMITO", "DOR_ABD", "FADIGA", "PERD_OLFT", "PERD_PALA"
]

COMORBIDITY_FEATURES = [
    "DIABETES", "CARDIOPATI", "PNEUMOPATI", "ASMA", "RENAL", "HEPATICA",
    "NEUROLOGIC", "IMUNODEPRE", "HEMATOLOGI", "SIND_DOWN", "OBES_IMC", "OUT_MORBI"
]

## 3. Feature standardization and binary mapping

Many variables in the source data are encoded inconsistently across files and years. This step converts raw categorical values into a unified binary representation suitable for downstream analysis.

In [ ]:
def map_binary_feature(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    mapped_numeric = numeric.map(lambda x: 1 if x == 1 else (0 if x == 2 else np.nan))

    if mapped_numeric.notna().mean() >= 0.2:
        return mapped_numeric.astype("Int8")

    text = series.astype(str).str.strip().str.upper()
    text = text.str.replace(r"\.0$", "", regex=True)
    text = text.replace({
        "SIM": "1",
        "S": "1",
        "YES": "1",
        "NAO": "2",
        "NÃO": "2",
        "N": "2",
        "NO": "2",
        "IGNORADO": "",
        "IGN": "",
        "NAO INFORMADO": "",
        "NÃO INFORMADO": "",
        "ND": "",
        "NA": "",
    })

    return text.map({"1": 1, "2": 0}).astype("Int8")

## 4. Raw data ingestion and year-wise processing

Each yearly file is processed in chunks to support large-scale data handling. At this stage, the notebook extracts relevant variables, harmonizes formats, filters invalid records, and saves intermediate parquet files for efficient downstream processing.

In [ ]:
def process_year(year: int) -> Path:
    source_path = CSV_FILES[year]
    output_path = INTERIM_DIR / f"early_{year}.parquet"

    required_columns = CORE_FEATURES + SYMPTOM_FEATURES + COMORBIDITY_FEATURES
    writer = None

    for chunk in pd.read_csv(
        source_path,
        sep=";",
        encoding="latin1",
        low_memory=False,
        chunksize=400_000,
    ):
        chunk.columns = [col.strip().upper() for col in chunk.columns]

        if "DT_SIN_PRI" not in chunk.columns:
            continue

        chunk["DT_SIN_PRI"] = pd.to_datetime(chunk["DT_SIN_PRI"], errors="coerce", dayfirst=True)
        age = pd.to_numeric(chunk["NU_IDADE_N"], errors="coerce")

        valid_mask = (
            chunk["DT_SIN_PRI"].notna()
            & chunk["EVOLUCAO"].isin([1, 2, 3])
            & chunk["CS_SEXO"].isin(["F", "M"])
            & age.between(0, 120)
        )

        data = chunk.loc[valid_mask, required_columns].copy()
        if data.empty:
            continue

        data["NU_IDADE_N"] = age.loc[valid_mask]
        data["Y_DEATH"] = chunk.loc[valid_mask, "EVOLUCAO"].isin([2, 3]).astype("Int8")
        data["YEAR"] = year
        data["SEX_FEMALE"] = (chunk.loc[valid_mask, "CS_SEXO"] == "F").astype("Int8")

        for col in SYMPTOM_FEATURES + COMORBIDITY_FEATURES:
            if col in data.columns:
                data[f"{col}_BIN"] = map_binary_feature(data[col])

        selected_columns = [
            "DT_SIN_PRI",
            "Y_DEATH",
            "NU_IDADE_N",
            "SEX_FEMALE",
            "YEAR",
            *[f"{col}_BIN" for col in SYMPTOM_FEATURES if col in data.columns],
            *[f"{col}_BIN" for col in COMORBIDITY_FEATURES if col in data.columns],
        ]

        table = pa.Table.from_pandas(data[selected_columns], preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema, compression="zstd", version="2.6")

        writer.write_table(table)

        del chunk, data, table
        gc.collect()

    if writer is not None:
        writer.close()

    return output_path


INTERIM_DIR.mkdir(parents=True, exist_ok=True)
interim_paths = [process_year(year) for year in YEARS]

df_interim = pd.concat([pd.read_parquet(path) for path in interim_paths], ignore_index=True)
combined_path = INTERIM_DIR / "early_2020_2022.parquet"
df_interim.to_parquet(combined_path, index=False, compression="zstd")

df_interim.shape

## 5. Column renaming and schema harmonization

To make the dataset easier to interpret and use in modeling, original source column names are converted into standardized English-style feature names. This also ensures a stable schema across all subsequent stages of the pipeline.

In [ ]:
COLUMN_MAP = {
    "DT_SIN_PRI": "symptom_onset_date",
    "NU_IDADE_N": "age_years",
    "SEX_FEMALE": "sex_female",
    "YEAR": "year",
    "Y_DEATH": "outcome_death",
    "FEBRE_BIN": "sym_fever",
    "TOSSE_BIN": "sym_cough",
    "GARGANTA_BIN": "sym_sore_throat",
    "DISPNEIA_BIN": "sym_dyspnea",
    "DESC_RESP_BIN": "sym_respiratory_distress",
    "SATURACAO_BIN": "sym_low_oxygen_saturation",
    "DIARREIA_BIN": "sym_diarrhea",
    "VOMITO_BIN": "sym_vomit",
    "FADIGA_BIN": "sym_fatigue",
    "PERD_OLFT_BIN": "sym_anosmia",
    "PERD_PALA_BIN": "sym_ageusia",
    "DOR_ABD_BIN": "sym_abdominal_pain",
    "DIABETES_BIN": "comorb_diabetes",
    "CARDIOPATI_BIN": "comorb_cardiovascular_disease",
    "PNEUMOPATI_BIN": "comorb_chronic_lung_disease",
    "ASMA_BIN": "comorb_asthma",
    "RENAL_BIN": "comorb_renal_disease",
    "HEPATICA_BIN": "comorb_hepatic_disease",
    "NEUROLOGIC_BIN": "comorb_neurologic_disorder",
    "IMUNODEPRE_BIN": "comorb_immunodeficiency",
    "HEMATOLOGI_BIN": "comorb_hematologic_disease",
    "SIND_DOWN_BIN": "comorb_down_syndrome",
    "OBES_IMC_BIN": "comorb_obesity_bmi",
    "OUT_MORBI_BIN": "comorb_other_morbidity",
}

df_std = pd.read_parquet(combined_path).rename(columns=COLUMN_MAP)

if "symptom_onset_date" in df_std.columns:
    df_std["symptom_onset_date"] = pd.to_datetime(df_std["symptom_onset_date"], errors="coerce")

for col in ["outcome_death", "sex_female"]:
    if col in df_std.columns:
        df_std[col] = df_std[col].astype("Int8")

std_path = INTERIM_DIR / "early_2020_2022_std.parquet"
df_std.to_parquet(std_path, index=False, compression="zstd")

df_std.shape

## 6. Initial dataset summary

After standardization, we compute basic descriptive statistics to verify dataset integrity, class balance, and missingness in the core variables. This provides a first sanity check before cleaning and feature enrichment.

In [ ]:
df = pd.read_parquet(std_path)

summary = {
    "n_total": len(df),
    "death_rate_overall": float(df["outcome_death"].mean()),
    "death_rate_by_year": df.groupby("year")["outcome_death"].mean().round(3).to_dict(),
    "missing_core": df[["symptom_onset_date", "age_years", "sex_female", "outcome_death"]]
        .isna()
        .mean()
        .round(4)
        .to_dict(),
}

summary

## 7. Data cleaning and validity checks

In this step, we enforce basic plausibility constraints on the standardized dataset. Invalid dates, impossible ages, inconsistent sex coding, and undefined outcomes are removed in order to obtain a clinically meaningful clean cohort.

In [ ]:
CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

df_clean = pd.read_parquet(std_path).copy()

df_clean["symptom_onset_date"] = pd.to_datetime(df_clean["symptom_onset_date"], errors="coerce")
df_clean["year"] = pd.to_numeric(df_clean["year"], errors="coerce").astype("Int16")
df_clean["age_years"] = pd.to_numeric(df_clean["age_years"], errors="coerce")

valid_mask = (
    df_clean["symptom_onset_date"].notna()
    & df_clean["age_years"].between(0, 120, inclusive="both")
    & df_clean["sex_female"].isin([0, 1, True, False])
    & df_clean["outcome_death"].isin([0, 1, True, False])
)

df_clean = df_clean.loc[valid_mask].copy()

binary_columns = [
    col for col in df_clean.columns
    if col.startswith("sym_") or col.startswith("comorb_")
] + ["sex_female", "outcome_death"]

for col in binary_columns:
    if col not in df_clean.columns:
        continue

    values = pd.to_numeric(df_clean[col], errors="coerce")
    values = values.where(values.isin([0.0, 1.0]), np.nan)
    df_clean[col] = values.astype("Int8")

df_clean["age_years"] = df_clean["age_years"].astype("Float32")

clean_path = CLEAN_DIR / "clean_2020_2022.parquet"
df_clean.to_parquet(clean_path, index=False)

df_clean.shape

## 8. Feature enrichment

Additional derived variables are created from the cleaned data, including calendar-based and age-based features. These transformations improve analytical flexibility while preserving the admission-time character of the dataset.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN_DIR = Path("./data/clean")
TABLES_DIR = Path("./outputs/tables")

input_path = CLEAN_DIR / "clean_2020_2022.parquet"
output_path = CLEAN_DIR / "clean_2020_2022_enriched.parquet"

df = pd.read_parquet(input_path).copy()

# Calendar features
dates = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["month"] = dates.dt.month.astype("Int8")
df["epi_week"] = dates.dt.isocalendar().week.astype("Int16")
df["epi_year"] = dates.dt.isocalendar().year.astype("Int16")

# Age features
age_bins = [0, 18, 40, 60, 75, 90, 120]
age_labels = ["age_00_17", "age_18_39", "age_40_59", "age_60_74", "age_75_89", "age_90p"]

df["age_bin"] = pd.cut(
    df["age_years"],
    bins=age_bins,
    labels=age_labels,
    right=True,
    include_lowest=True,
)

for label in age_labels:
    df[label] = (df["age_bin"] == label).astype("Int8")

df["age_sqrt"] = np.sqrt(df["age_years"]).astype("Float32")


def is_binary_feature(column_name: str) -> bool:
    return column_name in df.columns and str(df[column_name].dtype).startswith("Int")


def add_composite_feature(name: str, source_columns: list[str]) -> None:
    valid = pd.Series(False, index=df.index)
    positive = pd.Series(0, index=df.index, dtype="Int8")

    for col in source_columns:
        if is_binary_feature(col):
            valid |= df[col].notna()
            positive = ((positive == 1) | (df[col].fillna(0) == 1)).astype("Int8")

    df[name] = positive.where(valid, np.nan).astype("Int8")
    df[f"{name}_missing"] = df[name].isna().astype("Int8")


add_composite_feature(
    "sym_resp_comp_any",
    ["sym_dyspnea", "sym_respiratory_distress", "sym_low_oxygen_saturation"],
)
add_composite_feature(
    "sym_viral_any",
    ["sym_fever", "sym_cough", "sym_sore_throat"],
)
add_composite_feature(
    "sym_gi_any",
    ["sym_diarrhea", "sym_vomit", "sym_abdominal_pain"],
)
add_composite_feature(
    "sym_neuro_any",
    ["sym_anosmia", "sym_ageusia"],
)

# Weekly baseline risk
weekly = (
    df.groupby(["epi_year", "epi_week"])["outcome_death"]
    .mean()
    .reset_index()
    .sort_values(["epi_year", "epi_week"])
)

weekly["base_death_rate_expanding"] = (
    weekly.groupby("epi_year")["outcome_death"]
    .apply(lambda x: x.expanding(min_periods=2).mean())
    .values
)

weekly["base_death_rate_ma3"] = (
    weekly.groupby("epi_year")["outcome_death"]
    .apply(lambda x: x.rolling(window=3, min_periods=2).mean().shift(1))
    .values
)

df = df.merge(
    weekly[["epi_year", "epi_week", "base_death_rate_expanding", "base_death_rate_ma3"]],
    on=["epi_year", "epi_week"],
    how="left",
)

feature_dictionary = []

for label in age_labels:
    feature_dictionary.append(
        {"feature": label, "type": "binary", "note": "age bin indicator"}
    )

feature_dictionary.extend([
    {"feature": "age_sqrt", "type": "numeric", "note": "square root of age"},
    {"feature": "month", "type": "integer", "note": "month of symptom onset"},
    {"feature": "epi_week", "type": "integer", "note": "ISO epidemiological week"},
    {"feature": "epi_year", "type": "integer", "note": "ISO epidemiological year"},
])

for feature in ["sym_resp_comp_any", "sym_viral_any", "sym_gi_any", "sym_neuro_any"]:
    feature_dictionary.append(
        {"feature": feature, "type": "binary", "note": "composite symptom indicator"}
    )
    feature_dictionary.append(
        {"feature": f"{feature}_missing", "type": "binary", "note": "missingness flag"}
    )

for feature in ["base_death_rate_expanding", "base_death_rate_ma3"]:
    feature_dictionary.append(
        {"feature": feature, "type": "numeric", "note": "weekly baseline risk"}
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(feature_dictionary).to_csv(TABLES_DIR / "feature_dictionary.csv", index=False)

df.to_parquet(output_path, index=False)

df.shape

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

CLEAN_DIR = Path("./data/clean")

input_path = CLEAN_DIR / "clean_2020_2022_enriched.parquet"
output_path = CLEAN_DIR / "clean_2020_2022_modelready.parquet"

df = pd.read_parquet(input_path).copy()

binary_features = [
    col for col in df.columns
    if (col.startswith("sym_") or col.startswith("comorb_")) and not col.endswith("_missing")
]

binary_core = [col for col in ["sex_female", "outcome_death"] if col in df.columns]

for col in binary_features + binary_core:
    if col in df.columns:
        if f"{col}_missing" not in df.columns:
            df[f"{col}_missing"] = df[col].isna().astype("Int8")
        df[col] = df[col].fillna(0).astype("Int8")

continuous_features = [
    "age_years",
    "age_sqrt",
    "base_death_rate_expanding",
    "base_death_rate_ma3",
]

for col in continuous_features:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

scaled_features = [col for col in ["age_years", "age_sqrt"] if col in df.columns]

if scaled_features:
    scaler = StandardScaler()
    df[[f"{col}_z" for col in scaled_features]] = scaler.fit_transform(
        df[scaled_features].astype("float32")
    )

df.to_parquet(output_path, index=False)

df.shape

# 9. Exploratory data analysis

This section examines the structure of the final cleaned dataset before modeling. The goal is to understand class balance, temporal patterns, age effects, symptom distributions, and potential sources of dataset shift.

The exploratory analysis focuses on several clinically important aspects of the cohort:
- distribution of lethal outcomes across years;
- age-related risk gradients;
- symptom and comorbidity patterns;
- temporal and seasonal structure of observations.

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CLEAN_DIR = Path("./data/clean")
FIGS_DIR = Path("./outputs/figures")
FIGS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(CLEAN_DIR / "clean_2020_2022.parquet").copy()

In [ ]:
def get_year(frame: pd.DataFrame) -> pd.Series:
    return pd.to_numeric(frame["year"], errors="coerce").astype("Int16")


def get_symptom_columns(frame: pd.DataFrame) -> list[str]:
    return [col for col in frame.columns if col.startswith("sym_") and not col.endswith("_missing")]


def get_epi_year_week(frame: pd.DataFrame, date_col: str = "symptom_onset_date") -> tuple[pd.Series, pd.Series]:
    dates = pd.to_datetime(frame[date_col], errors="coerce")
    iso = dates.dt.isocalendar()
    return iso["year"].astype("Int16"), iso["week"].astype("Int16")


def to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

In [ ]:
plt.figure(figsize=(8, 5))

for year in [2020, 2021, 2022]:
    ages = df.loc[get_year(df) == year, "age_years"].dropna().astype(float)
    bins = np.arange(0, 121, 2)
    counts, edges = np.histogram(ages, bins=bins, density=True)
    centers = (edges[:-1] + edges[1:]) / 2
    plt.plot(centers, counts, label=str(year))

plt.xlabel("Age, years")
plt.ylabel("Density")
plt.title("Age distribution by year")
plt.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR / "age_hist_by_year.png", dpi=200)
plt.show()

In [ ]:
age_bins = [0, 18, 40, 60, 75, 90, 200]
age_labels = ["0–17", "18–39", "40–59", "60–74", "75–89", "90+"]

plot_df = df.copy()
plot_df["age_group"] = pd.cut(
    plot_df["age_years"].astype(float),
    bins=age_bins,
    labels=age_labels,
    right=False,
)

rates_by_year = {}
for year in [2020, 2021, 2022]:
    subset = plot_df[get_year(plot_df) == year]
    rates = subset.groupby("age_group")["outcome_death"].mean()
    rates_by_year[year] = rates.reindex(age_labels).values

plt.figure(figsize=(8, 5))
x = np.arange(len(age_labels))

for year in [2020, 2021, 2022]:
    plt.plot(x, rates_by_year[year], marker="o", label=str(year))

plt.xticks(x, age_labels)
plt.ylim(0, 1)
plt.xlabel("Age group")
plt.ylabel("Mortality rate")
plt.title("Mortality by age group and year")
plt.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR / "mortality_by_agebins_by_year.png", dpi=200)
plt.show()

In [ ]:
symptom_cols = get_symptom_columns(df)
comorbidity_cols = [col for col in df.columns if col.startswith("comorb_") and not col.endswith("_missing")]
feature_cols = symptom_cols + comorbidity_cols

missing_matrix = []

for col in feature_cols:
    shares = df.groupby(get_year(df))[col].apply(lambda x: x.isna().mean())
    missing_matrix.append([
        shares.get(2020, np.nan),
        shares.get(2021, np.nan),
        shares.get(2022, np.nan),
    ])

missing_df = pd.DataFrame(
    missing_matrix,
    index=feature_cols,
    columns=[2020, 2021, 2022],
)

plt.figure(figsize=(8, max(4, 0.22 * len(missing_df))))
plt.imshow(missing_df.values, aspect="auto", interpolation="nearest")
plt.colorbar(label="Missing share")
plt.yticks(range(len(missing_df.index)), missing_df.index)
plt.xticks(range(3), missing_df.columns)
plt.title("Missingness by feature and year")
plt.tight_layout()
plt.savefig(FIGS_DIR / "missingness_heatmap_by_year.png", dpi=200)
plt.show()

In [ ]:
plot_df = df[["symptom_onset_date", "outcome_death"]].dropna().copy()
plot_df["symptom_onset_date"] = pd.to_datetime(plot_df["symptom_onset_date"], errors="coerce")

iso = plot_df["symptom_onset_date"].dt.isocalendar()
plot_df["epi_year"] = iso["year"].astype(int)
plot_df["epi_week"] = iso["week"].astype(int)

weekly = (
    plot_df.groupby(["epi_year", "epi_week"])["outcome_death"]
    .mean()
    .reset_index()
    .sort_values(["epi_year", "epi_week"])
)

weekly = weekly[(weekly["epi_year"] >= 2020) & (weekly["epi_year"] <= 2022)].copy()

weekly["expanding_mean_lag1"] = (
    weekly.groupby("epi_year")["outcome_death"]
    .apply(lambda x: x.expanding(min_periods=2).mean().shift(1))
    .values
)

weekly["ma3_lag1"] = (
    weekly.groupby("epi_year")["outcome_death"]
    .apply(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
    .values
)

weekly["t"] = (weekly["epi_year"] - weekly["epi_year"].min()) * 60 + weekly["epi_week"]

plt.figure(figsize=(9, 5))
plt.plot(weekly["t"], weekly["outcome_death"], label="Weekly mortality")
plt.plot(weekly["t"], weekly["expanding_mean_lag1"], label="Expanding mean (lag 1)")
plt.plot(weekly["t"], weekly["ma3_lag1"], label="Moving average 3 (lag 1)")

plt.xlabel("Pseudo-week (2020–2022)")
plt.ylabel("Mortality rate")
plt.title("Weekly mortality with non-leaky smoothing")
plt.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR / "weekly_mortality_with_smoothing.png", dpi=200)
plt.show()

In [ ]:
class_balance = (
    df.groupby(get_year(df))["outcome_death"]
    .mean()
    .reindex([2020, 2021, 2022])
)

plt.figure(figsize=(6, 5))
plt.bar(class_balance.index.astype(str), class_balance.values)

plt.ylim(0, 1)
plt.xlabel("Year")
plt.ylabel("Mortality rate")
plt.title("Class balance by year")
plt.tight_layout()
plt.savefig(FIGS_DIR / "class_balance_by_year.png", dpi=200)
plt.show()

In [ ]:
plot_df = df[["symptom_onset_date"]].dropna().copy()
plot_df["symptom_onset_date"] = pd.to_datetime(plot_df["symptom_onset_date"], errors="coerce")

iso = plot_df["symptom_onset_date"].dt.isocalendar()
plot_df["epi_year"] = iso["year"].astype(int)
plot_df["epi_week"] = iso["week"].astype(int)

weekly_counts = (
    plot_df.groupby(["epi_year", "epi_week"])
    .size()
    .reset_index(name="n")
)

weekly_counts = weekly_counts[(weekly_counts["epi_year"] >= 2020) & (weekly_counts["epi_year"] <= 2022)].copy()
weekly_counts["t"] = (weekly_counts["epi_year"] - weekly_counts["epi_year"].min()) * 60 + weekly_counts["epi_week"]

plt.figure(figsize=(9, 5))
plt.plot(weekly_counts["t"], weekly_counts["n"])

plt.xlabel("Pseudo-week (2020–2022)")
plt.ylabel("Number of admissions")
plt.title("Weekly admissions")
plt.tight_layout()
plt.savefig(FIGS_DIR / "weekly_admissions.png", dpi=200)
plt.show()

In [ ]:
plot_df = df.copy()
plot_df["sex_label"] = plot_df["sex_female"].map({1: "F", 0: "M"}).astype("string")

rates = (
    plot_df.groupby([get_year(plot_df), "sex_label"])["outcome_death"]
    .mean()
    .reset_index()
    .rename(columns={"year": "plot_year"})
)

years = [2020, 2021, 2022]
sexes = ["M", "F"]

values_m = []
values_f = []

for year in years:
    value_m = rates.loc[(rates["plot_year"] == year) & (rates["sex_label"] == "M"), "outcome_death"]
    value_f = rates.loc[(rates["plot_year"] == year) & (rates["sex_label"] == "F"), "outcome_death"]

    values_m.append(float(value_m.iloc[0]) if len(value_m) else np.nan)
    values_f.append(float(value_f.iloc[0]) if len(value_f) else np.nan)

x = np.arange(len(years))
width = 0.35

plt.figure(figsize=(7, 5))
plt.bar(x - width / 2, values_m, width, label="M")
plt.bar(x + width / 2, values_f, width, label="F")

plt.xticks(x, [str(year) for year in years])
plt.ylim(0, 1)
plt.xlabel("Year")
plt.ylabel("Mortality rate")
plt.title("Mortality by sex and year")
plt.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR / "mortality_by_sex_year.png", dpi=200)
plt.show()

In [ ]:
plot_df = df[["age_years", "outcome_death"]].dropna().copy()

age_discharge = to_numeric(plot_df.loc[plot_df["outcome_death"] == 0, "age_years"]).values
age_death = to_numeric(plot_df.loc[plot_df["outcome_death"] == 1, "age_years"]).values

plt.figure(figsize=(7, 5))
plt.violinplot([age_discharge, age_death], showmeans=True, showextrema=False, showmedians=False)

plt.xticks([1, 2], ["Discharge (0)", "Death (1)"])
plt.ylabel("Age, years")
plt.title("Age distribution by outcome")
plt.tight_layout()
plt.savefig(FIGS_DIR / "age_violin_by_outcome.png", dpi=200)
plt.show()

In [ ]:
binary_features = [
    col for col in df.columns
    if (col.startswith("sym_") or col.startswith("comorb_") or col == "sex_female")
    and not col.endswith("_missing")
]

def compute_log_or(feature: str, target: str = "outcome_death") -> tuple[float, float, float, int]:
    subset = df[[feature, target]].dropna()
    if subset.empty:
        raise ValueError("Empty 2x2 table")

    x = subset[feature].astype(int)
    y = subset[target].astype(int)

    a = int(((x == 1) & (y == 1)).sum())
    b = int(((x == 1) & (y == 0)).sum())
    c = int(((x == 0) & (y == 1)).sum())
    d = int(((x == 0) & (y == 0)).sum())

    a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5

    log_or = math.log((a * d) / (b * c))
    se = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    ci_low = log_or - 1.96 * se
    ci_high = log_or + 1.96 * se

    return log_or, ci_low, ci_high, int(len(subset))


rows = []

for feature in binary_features:
    try:
        log_or, ci_low, ci_high, n = compute_log_or(feature)
        rows.append((feature, log_or, ci_low, ci_high, n))
    except Exception:
        pass

or_df = pd.DataFrame(rows, columns=["feature", "log_or", "ci_low", "ci_high", "n"]).dropna()
or_df["abs_log_or"] = or_df["log_or"].abs()

top_df = (
    or_df.sort_values(["abs_log_or", "n"], ascending=[False, False])
    .head(20)
    .sort_values("log_or")
)

plt.figure(figsize=(8, max(6, 0.4 * len(top_df))))
ypos = np.arange(len(top_df))
xerr_low = top_df["log_or"] - top_df["ci_low"]
xerr_high = top_df["ci_high"] - top_df["log_or"]

plt.errorbar(top_df["log_or"], ypos, xerr=[xerr_low, xerr_high], fmt="o")
plt.axvline(0.0, linestyle="--")
plt.yticks(ypos, top_df["feature"])
plt.xlabel("log(OR) with 95% CI")
plt.title("Top binary predictors by absolute log(OR)")
plt.tight_layout()
plt.savefig(FIGS_DIR / "univariate_log_or_forest_top20.png", dpi=200)
plt.show()

# 10. Model-ready dataset construction

This section prepares the final machine-learning inputs. The dataset is restricted to variables available at or near admission, the target variable is defined, and temporal splits are created for training, validation, and independent testing.

The modeling design follows a temporal evaluation logic. Earlier epidemic years are used for training and validation, while the later period is reserved for independent testing in order to reflect realistic distribution shift.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

CLEAN_DIR = Path("./data/clean")
MODELREADY_DIR = Path("./data/modelready")
META_DIR = Path("./outputs/meta")

MODELREADY_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(CLEAN_DIR / "clean_2020_2022_modelready.parquet").copy()

df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int16")
df = df[df["year"].isin([2020, 2021, 2022])].copy()

target = pd.to_numeric(df["outcome_death"], errors="coerce").astype("Int8")

drop_columns = {"outcome_death", "symptom_onset_date", "year"}
feature_columns = [col for col in df.columns if col not in drop_columns]

binary_features = [
    col for col in feature_columns
    if col.startswith("sym_") or col.startswith("comorb_") or col == "sex_female"
]

assert all(df[col].dropna().isin([0, 1]).all() for col in binary_features)

for col in [c for c in binary_features if not c.endswith("_missing")]:
    assert f"{col}_missing" in df.columns

train_pool = df[df["year"].isin([2020, 2021])].copy()
test_df = df[df["year"] == 2022].copy()

validation_cutoff = train_pool["symptom_onset_date"].quantile(0.75)

train_df = train_pool[train_pool["symptom_onset_date"] < validation_cutoff].copy()
val_df = train_pool[train_pool["symptom_onset_date"] >= validation_cutoff].copy()

assert train_df["symptom_onset_date"].max() < test_df["symptom_onset_date"].min()

X_train = train_df[feature_columns].copy()
y_train = target.loc[train_df.index].astype("Int8")

X_val = val_df[feature_columns].copy()
y_val = target.loc[val_df.index].astype("Int8")

X_test = test_df[feature_columns].copy()
y_test = target.loc[test_df.index].astype("Int8")

X_train.to_parquet(MODELREADY_DIR / "X_train.parquet")
y_train.to_frame("y").to_parquet(MODELREADY_DIR / "y_train.parquet")

X_val.to_parquet(MODELREADY_DIR / "X_val.parquet")
y_val.to_frame("y").to_parquet(MODELREADY_DIR / "y_val.parquet")

X_test.to_parquet(MODELREADY_DIR / "X_test.parquet")
y_test.to_frame("y").to_parquet(MODELREADY_DIR / "y_test.parquet")

split_manifest = {
    "paths": {
        "X_train": str(MODELREADY_DIR / "X_train.parquet"),
        "y_train": str(MODELREADY_DIR / "y_train.parquet"),
        "X_val": str(MODELREADY_DIR / "X_val.parquet"),
        "y_val": str(MODELREADY_DIR / "y_val.parquet"),
        "X_test": str(MODELREADY_DIR / "X_test.parquet"),
        "y_test": str(MODELREADY_DIR / "y_test.parquet"),
    },
    "features": feature_columns,
    "sizes": {
        "train": int(len(X_train)),
        "val": int(len(X_val)),
        "test": int(len(X_test)),
    },
    "time_split": {
        "train_years": [2020, 2021],
        "test_year": 2022,
        "val_cutoff": validation_cutoff.isoformat() if pd.notna(validation_cutoff) else None,
    },
}

with open(META_DIR / "split_manifest.json", "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, ensure_ascii=False, indent=2)

split_manifest["sizes"]

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

from xgboost import XGBClassifier
import lightgbm as lgb

In [ ]:
MODELREADY_DIR = Path("./data/modelready")
META_DIR = Path("./outputs/meta")
FIGS_DIR = Path("./outputs/figures")
TABLES_DIR = Path("./outputs/tables")

FIGS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

with open(META_DIR / "split_manifest.json", "r", encoding="utf-8") as f:
    split_manifest = json.load(f)

feature_columns = split_manifest["features"]

X_train = pd.read_parquet(MODELREADY_DIR / "X_train.parquet")[feature_columns].copy()
y_train = pd.read_parquet(MODELREADY_DIR / "y_train.parquet")["y"].astype("int8").copy()

X_val = pd.read_parquet(MODELREADY_DIR / "X_val.parquet")[feature_columns].copy()
y_val = pd.read_parquet(MODELREADY_DIR / "y_val.parquet")["y"].astype("int8").copy()

X_test = pd.read_parquet(MODELREADY_DIR / "X_test.parquet")[feature_columns].copy()
y_test = pd.read_parquet(MODELREADY_DIR / "y_test.parquet")["y"].astype("int8").copy()

X_train.shape, X_val.shape, X_test.shape

In [ ]:
for frame in (X_train, X_val, X_test):
    if "age_bin" in frame.columns:
        frame.drop(columns=["age_bin"], inplace=True)

fill_values = {}

binary_like = [
    col for col in X_train.columns
    if col.startswith(("sym_", "comorb_")) or col == "sex_female" or col.endswith("_missing")
]

for col in binary_like:
    fill_values[col] = 0.0

for col in X_train.columns:
    if col not in fill_values:
        median_value = np.nanmedian(pd.to_numeric(X_train[col], errors="coerce"))
        fill_values[col] = float(0.0 if np.isnan(median_value) else median_value)


def apply_fill_values(frame: pd.DataFrame, value_map: dict) -> pd.DataFrame:
    output = frame.copy()
    for col, value in value_map.items():
        output[col] = pd.to_numeric(output[col], errors="coerce").astype("float32").fillna(value)
    return output


X_train = apply_fill_values(X_train, fill_values)
X_val = apply_fill_values(X_val, fill_values)
X_test = apply_fill_values(X_test, fill_values)

for obj in (X_train, X_val, X_test, y_train, y_val, y_test):
    obj.reset_index(drop=True, inplace=True)

assert not np.isnan(X_train.values).any()
assert not np.isnan(X_val.values).any()
assert not np.isnan(X_test.values).any()

In [ ]:
def logit_transform(probabilities: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    probabilities = np.clip(probabilities, eps, 1 - eps)
    return np.log(probabilities / (1 - probabilities))


def calibration_slope_intercept(y_true: np.ndarray, y_prob: np.ndarray) -> tuple[float, float]:
    x = logit_transform(y_prob).reshape(-1, 1)
    model = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    model.fit(x, y_true)
    return float(model.coef_[0, 0]), float(model.intercept_[0])


def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, split_name: str) -> dict:
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    slope, intercept = calibration_slope_intercept(y_true, y_prob)

    return {
        "split": split_name,
        "AUROC": auroc,
        "PR_AUC": auprc,
        "Brier": brier,
        "CalibrationSlope": slope,
        "CalibrationIntercept": intercept,
    }


def fit_isotonic_calibration(
    val_prob: np.ndarray,
    y_val_true: np.ndarray,
    target_prob: np.ndarray,
) -> tuple[IsotonicRegression, np.ndarray]:
    calibrator = IsotonicRegression(out_of_bounds="clip")
    calibrator.fit(val_prob, y_val_true)
    calibrated_prob = calibrator.predict(target_prob)
    return calibrator, calibrated_prob


def save_current_figure(path: Path) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.show()
    plt.close()

In [ ]:
def plot_reliability_curve(y_true: np.ndarray, y_prob: np.ndarray, title: str, filename: str) -> None:
    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=20, strategy="quantile")

    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], "--", linewidth=1)
    plt.plot(mean_pred, frac_pos, "o-")
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed frequency")
    plt.title(title)

    save_current_figure(FIGS_DIR / filename)


def plot_roc_pr_curves(y_true: np.ndarray, y_prob: np.ndarray, prefix: str) -> None:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)

    plt.figure()
    plt.plot(fpr, tpr)
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(f"ROC — {prefix}")
    save_current_figure(FIGS_DIR / f"roc_{prefix}.png")

    plt.figure()
    plt.plot(recall, precision)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"PR — {prefix}")
    save_current_figure(FIGS_DIR / f"pr_{prefix}.png")

In [ ]:
def decision_curve(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    thresholds: np.ndarray = np.linspace(0.01, 0.99, 99),
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)

    nb_model = []
    nb_all = []
    nb_none = []

    for threshold in thresholds:
        predicted = (y_prob >= threshold).astype(int)

        true_positive = ((predicted == 1) & (y_true == 1)).sum()
        false_positive = ((predicted == 1) & (y_true == 0)).sum()

        nb_model.append((true_positive / n) - (false_positive / n) * (threshold / (1 - threshold)))
        nb_all.append((y_true == 1).sum() / n - (y_true == 0).sum() / n * (threshold / (1 - threshold)))
        nb_none.append(0.0)

    return thresholds, np.array(nb_model), np.array(nb_all), np.array(nb_none)


def plot_decision_curve(y_true: np.ndarray, y_prob: np.ndarray, title: str, filename: str) -> None:
    thresholds, nb_model, nb_all, nb_none = decision_curve(y_true, y_prob)

    plt.figure(figsize=(6, 4))
    plt.plot(thresholds, nb_model, label="Model")
    plt.plot(thresholds, nb_all, "--", label="Treat all")
    plt.plot(thresholds, nb_none, ":", label="Treat none")
    plt.xlabel("Threshold probability")
    plt.ylabel("Net benefit per patient")
    plt.title(title)
    plt.legend()

    save_current_figure(FIGS_DIR / filename)

# 11. Baseline and individual predictive models

We train several individual machine-learning models and evaluate them on validation and test sets. In addition to discrimination metrics, probability calibration is assessed because reliable clinical risk estimates are essential for decision support.

The initial modeling stage includes strong baseline and boosting-based approaches. Each model is evaluated not only by ranking performance, but also by the quality of calibrated probability estimates.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

MODEL_NAME = "LR"

prediction_path = TABLES_DIR / f"preds_{MODEL_NAME}_test_iso.parquet"
metrics_path = TABLES_DIR / "models_metrics.csv"

lr_model = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    max_iter=2000,
)

lr_model.fit(X_train, y_train)

val_prob = lr_model.predict_proba(X_val)[:, 1]
test_prob = lr_model.predict_proba(X_test)[:, 1]

isotonic_model, test_prob_iso = fit_isotonic_calibration(val_prob, y_val, test_prob)

metrics_row = {
    "model": MODEL_NAME,
    "split": "test_iso",
    "AUROC": roc_auc_score(y_test, test_prob_iso),
    "PR_AUC": average_precision_score(y_test, test_prob_iso),
    "Brier": brier_score_loss(y_test, test_prob_iso),
}

metrics_df = pd.DataFrame([metrics_row])

if metrics_path.exists():
    previous_metrics = pd.read_csv(metrics_path)
    metrics_df = pd.concat([previous_metrics, metrics_df], ignore_index=True)

metrics_df.to_csv(metrics_path, index=False)

plot_reliability_curve(
    y_test,
    test_prob_iso,
    "Calibration curve — Logistic Regression",
    "calibration_lr_test.png",
)

plot_roc_pr_curves(
    y_test,
    test_prob_iso,
    "lr_test",
)

plot_decision_curve(
    y_test,
    test_prob_iso,
    "Decision curve — Logistic Regression",
    "dca_lr_test.png",
)

pd.DataFrame({"p_test_iso": test_prob_iso}).to_parquet(prediction_path, index=False)

pd.DataFrame([metrics_row])

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

MODEL_NAME = "LR_EN"

prediction_path = TABLES_DIR / f"preds_{MODEL_NAME}_test_iso.parquet"
metrics_path = TABLES_DIR / "models_metrics.csv"

lren_model = LogisticRegression(
    solver="saga",
    penalty="elasticnet",
    l1_ratio=0.2,
    C=1.0,
    max_iter=4000,
    n_jobs=-1,
    random_state=42,
)

lren_model.fit(X_train, y_train)

val_prob = lren_model.predict_proba(X_val)[:, 1]
test_prob = lren_model.predict_proba(X_test)[:, 1]

isotonic_model, test_prob_iso = fit_isotonic_calibration(val_prob, y_val, test_prob)

metrics_row = {
    "model": MODEL_NAME,
    "split": "test_iso",
    "AUROC": roc_auc_score(y_test, test_prob_iso),
    "PR_AUC": average_precision_score(y_test, test_prob_iso),
    "Brier": brier_score_loss(y_test, test_prob_iso),
}

metrics_df = pd.DataFrame([metrics_row])

if metrics_path.exists():
    previous_metrics = pd.read_csv(metrics_path)
    previous_metrics = previous_metrics[
        ~((previous_metrics["model"] == MODEL_NAME) & (previous_metrics["split"] == "test_iso"))
    ]
    metrics_df = pd.concat([previous_metrics, metrics_df], ignore_index=True)

metrics_df.to_csv(metrics_path, index=False)

plot_reliability_curve(
    y_test,
    test_prob_iso,
    "Calibration curve — Elastic Net Logistic Regression",
    "calibration_lren_test.png",
)

plot_roc_pr_curves(
    y_test,
    test_prob_iso,
    "lren_test",
)

plot_decision_curve(
    y_test,
    test_prob_iso,
    "Decision curve — Elastic Net Logistic Regression",
    "dca_lren_test.png",
)

pd.DataFrame({"p_test_iso": test_prob_iso}).to_parquet(prediction_path, index=False)

pd.DataFrame([metrics_row])

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

MODEL_NAME = "HGB"

val_prediction_path = TABLES_DIR / f"preds_{MODEL_NAME}_val_iso.parquet"
test_prediction_path = TABLES_DIR / f"preds_{MODEL_NAME}_test_iso.parquet"
metrics_path = TABLES_DIR / "models_metrics.csv"

hgb_model = HistGradientBoostingClassifier(
    max_depth=None,
    learning_rate=0.03,
    max_iter=600,
    l2_regularization=0.1,
    random_state=42,
)

hgb_model.fit(X_train, y_train)

val_prob_raw = hgb_model.predict_proba(X_val)[:, 1]
test_prob_raw = hgb_model.predict_proba(X_test)[:, 1]

isotonic_model, val_prob_iso = fit_isotonic_calibration(val_prob_raw, y_val, val_prob_raw)
test_prob_iso = isotonic_model.predict(test_prob_raw)

metrics_row = {
    "model": MODEL_NAME,
    "split": "test_iso",
    "AUROC": roc_auc_score(y_test, test_prob_iso),
    "PR_AUC": average_precision_score(y_test, test_prob_iso),
    "Brier": brier_score_loss(y_test, test_prob_iso),
}

metrics_df = pd.DataFrame([metrics_row])

if metrics_path.exists():
    previous_metrics = pd.read_csv(metrics_path)
    previous_metrics = previous_metrics[
        ~((previous_metrics["model"] == MODEL_NAME) & (previous_metrics["split"] == "test_iso"))
    ]
    metrics_df = pd.concat([previous_metrics, metrics_df], ignore_index=True)

metrics_df.to_csv(metrics_path, index=False)

plot_reliability_curve(
    y_test,
    test_prob_iso,
    "Calibration curve — HistGradientBoosting",
    "calibration_hgb_test.png",
)

plot_roc_pr_curves(
    y_test,
    test_prob_iso,
    "hgb_test",
)

plot_decision_curve(
    y_test,
    test_prob_iso,
    "Decision curve — HistGradientBoosting",
    "dca_hgb_test.png",
)

pd.DataFrame({"p": val_prob_iso}).to_parquet(val_prediction_path, index=False)
pd.DataFrame({"p": test_prob_iso}).to_parquet(test_prediction_path, index=False)

pd.DataFrame([metrics_row])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

MODEL_NAME = "RF"

prediction_path = TABLES_DIR / f"preds_{MODEL_NAME}_test_iso.parquet"
metrics_path = TABLES_DIR / "models_metrics.csv"
importance_csv_path = TABLES_DIR / f"feat_importance_{MODEL_NAME}.csv"
importance_plot_path = FIGS_DIR / f"feat_importance_{MODEL_NAME}.png"

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=42,
    oob_score=False,
)

rf_model.fit(X_train, y_train)

val_prob = rf_model.predict_proba(X_val)[:, 1]
test_prob = rf_model.predict_proba(X_test)[:, 1]

isotonic_model, test_prob_iso = fit_isotonic_calibration(val_prob, y_val, test_prob)

metrics_row = {
    "model": MODEL_NAME,
    "split": "test_iso",
    "AUROC": roc_auc_score(y_test, test_prob_iso),
    "PR_AUC": average_precision_score(y_test, test_prob_iso),
    "Brier": brier_score_loss(y_test, test_prob_iso),
}

metrics_df = pd.DataFrame([metrics_row])

if metrics_path.exists():
    previous_metrics = pd.read_csv(metrics_path)
    previous_metrics = previous_metrics[
        ~((previous_metrics["model"] == MODEL_NAME) & (previous_metrics["split"] == "test_iso"))
    ]
    metrics_df = pd.concat([previous_metrics, metrics_df], ignore_index=True)

metrics_df.to_csv(metrics_path, index=False)

plot_reliability_curve(
    y_test,
    test_prob_iso,
    "Calibration curve — Random Forest",
    "calibration_rf_test.png",
)

plot_roc_pr_curves(
    y_test,
    test_prob_iso,
    "rf_test",
)

plot_decision_curve(
    y_test,
    test_prob_iso,
    "Decision curve — Random Forest",
    "dca_rf_test.png",
)

pd.DataFrame({"p_test_iso": test_prob_iso}).to_parquet(prediction_path, index=False)

feature_importance = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importance": rf_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance.to_csv(importance_csv_path, index=False)

top_features = feature_importance.head(30).sort_values("importance", ascending=True)

plt.figure(figsize=(7, 8))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest — Top 30 Feature Importances")
plt.tight_layout()
plt.savefig(importance_plot_path, dpi=200)
plt.show()

pd.DataFrame([metrics_row])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

MODEL_NAME = "LGBM"

val_prediction_path = TABLES_DIR / f"preds_{MODEL_NAME}_val_iso.parquet"
test_prediction_path = TABLES_DIR / f"preds_{MODEL_NAME}_test_iso.parquet"
metrics_path = TABLES_DIR / "models_metrics.csv"
importance_csv_path = TABLES_DIR / f"feat_importance_{MODEL_NAME}.csv"
importance_plot_path = FIGS_DIR / f"feat_importance_{MODEL_NAME}.png"

lgbm_model = lgb.LGBMClassifier(
    objective="binary",
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=60,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=0.0,
    n_estimators=1000,
    random_state=42,
    n_jobs=-1,
)

lgbm_model.fit(X_train, y_train)

val_prob_raw = lgbm_model.predict_proba(X_val)[:, 1]
test_prob_raw = lgbm_model.predict_proba(X_test)[:, 1]

isotonic_model, val_prob_iso = fit_isotonic_calibration(val_prob_raw, y_val, val_prob_raw)
test_prob_iso = isotonic_model.predict(test_prob_raw)

metrics_row = {
    "model": MODEL_NAME,
    "split": "test_iso",
    "AUROC": roc_auc_score(y_test, test_prob_iso),
    "PR_AUC": average_precision_score(y_test, test_prob_iso),
    "Brier": brier_score_loss(y_test, test_prob_iso),
}

metrics_df = pd.DataFrame([metrics_row])

if metrics_path.exists():
    previous_metrics = pd.read_csv(metrics_path)
    previous_metrics = previous_metrics[
        ~((previous_metrics["model"] == MODEL_NAME) & (previous_metrics["split"] == "test_iso"))
    ]
    metrics_df = pd.concat([previous_metrics, metrics_df], ignore_index=True)

metrics_df.to_csv(metrics_path, index=False)

plot_reliability_curve(
    y_test,
    test_prob_iso,
    "Calibration curve — LightGBM",
    "calibration_lgbm_test.png",
)

plot_roc_pr_curves(
    y_test,
    test_prob_iso,
    "lgbm_test",
)

plot_decision_curve(
    y_test,
    test_prob_iso,
    "Decision curve — LightGBM",
    "dca_lgbm_test.png",
)

feature_importance = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importance_gain": lgbm_model.booster_.feature_importance(importance_type="gain"),
    })
    .sort_values("importance_gain", ascending=False)
    .reset_index(drop=True)
)

feature_importance.to_csv(importance_csv_path, index=False)

top_features = feature_importance.head(30).sort_values("importance_gain", ascending=True)

plt.figure(figsize=(7, 8))
plt.barh(top_features["feature"], top_features["importance_gain"])
plt.xlabel("Importance (gain)")
plt.ylabel("Feature")
plt.title("LightGBM — Top 30 Feature Importances")
plt.tight_layout()
plt.savefig(importance_plot_path, dpi=200)
plt.show()

pd.DataFrame({"p": val_prob_iso}).to_parquet(val_prediction_path, index=False)
pd.DataFrame({"p": test_prob_iso}).to_parquet(test_prediction_path, index=False)

pd.DataFrame([metrics_row])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

MODEL_NAME = "XGB"

val_prediction_path = TABLES_DIR / f"preds_{MODEL_NAME}_val_iso.parquet"
test_prediction_path = TABLES_DIR / f"preds_{MODEL_NAME}_test_iso.parquet"
metrics_path = TABLES_DIR / "models_metrics.csv"
importance_csv_path = TABLES_DIR / f"feat_importance_{MODEL_NAME}.csv"
importance_plot_path = FIGS_DIR / f"feat_importance_{MODEL_NAME}.png"

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(X_train.columns))
dval = xgb.DMatrix(X_val, label=y_val, feature_names=list(X_val.columns))
dtest = xgb.DMatrix(X_test, feature_names=list(X_test.columns))

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "eta": 0.03,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 1.0,
    "tree_method": "hist",
    "grow_policy": "lossguide",
    "seed": 42,
}

xgb_model = xgb.train(
    params=xgb_params,
    dtrain=dtrain,
    num_boost_round=2000,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=100,
    verbose_eval=False,
)

best_iteration = getattr(xgb_model, "best_iteration", None)

if best_iteration is not None:
    val_prob_raw = xgb_model.predict(dval, iteration_range=(0, best_iteration + 1))
    test_prob_raw = xgb_model.predict(dtest, iteration_range=(0, best_iteration + 1))
else:
    val_prob_raw = xgb_model.predict(dval)
    test_prob_raw = xgb_model.predict(dtest)

isotonic_model, val_prob_iso = fit_isotonic_calibration(val_prob_raw, y_val, val_prob_raw)
test_prob_iso = isotonic_model.predict(test_prob_raw)

metrics_row = {
    "model": MODEL_NAME,
    "split": "test_iso",
    "AUROC": roc_auc_score(y_test, test_prob_iso),
    "PR_AUC": average_precision_score(y_test, test_prob_iso),
    "Brier": brier_score_loss(y_test, test_prob_iso),
}

metrics_df = pd.DataFrame([metrics_row])

if metrics_path.exists():
    previous_metrics = pd.read_csv(metrics_path)
    previous_metrics = previous_metrics[
        ~((previous_metrics["model"] == MODEL_NAME) & (previous_metrics["split"] == "test_iso"))
    ]
    metrics_df = pd.concat([previous_metrics, metrics_df], ignore_index=True)

metrics_df.to_csv(metrics_path, index=False)

plot_reliability_curve(
    y_test,
    test_prob_iso,
    "Calibration curve — XGBoost",
    "calibration_xgb_test.png",
)

plot_roc_pr_curves(
    y_test,
    test_prob_iso,
    "xgb_test",
)

plot_decision_curve(
    y_test,
    test_prob_iso,
    "Decision curve — XGBoost",
    "dca_xgb_test.png",
)

importance_scores = xgb_model.get_score(importance_type="gain")

feature_importance = (
    pd.DataFrame({
        "feature": list(importance_scores.keys()),
        "importance_gain": list(importance_scores.values()),
    })
    .sort_values("importance_gain", ascending=False)
    .reset_index(drop=True)
)

feature_importance.to_csv(importance_csv_path, index=False)

top_features = feature_importance.head(30).sort_values("importance_gain", ascending=True)

plt.figure(figsize=(7, 8))
plt.barh(top_features["feature"], top_features["importance_gain"])
plt.xlabel("Importance (gain)")
plt.ylabel("Feature")
plt.title("XGBoost — Top 30 Feature Importances")
plt.tight_layout()
plt.savefig(importance_plot_path, dpi=200)
plt.show()

pd.DataFrame({"p": val_prob_iso}).to_parquet(val_prediction_path, index=False)
pd.DataFrame({"p": test_prob_iso}).to_parquet(test_prediction_path, index=False)

pd.DataFrame([metrics_row])

# 12. Weighted ensemble of top-performing boosting models

To improve robustness and predictive performance, the strongest individual boosting models are combined into a weighted ensemble. The ensemble is then calibrated and compared with single-model alternatives.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

MODELREADY_DIR = Path("./data/modelready")
TABLES_DIR = Path("./outputs/tables")
FIGS_DIR = Path("./outputs/figures")

y_val = pd.read_parquet(MODELREADY_DIR / "y_val.parquet")["y"].astype("int8").to_numpy()
y_test = pd.read_parquet(MODELREADY_DIR / "y_test.parquet")["y"].astype("int8").to_numpy()

required_paths = {
    "lgbm_val": TABLES_DIR / "preds_LGBM_val_iso.parquet",
    "lgbm_test": TABLES_DIR / "preds_LGBM_test_iso.parquet",
    "hgb_val": TABLES_DIR / "preds_HGB_val_iso.parquet",
    "hgb_test": TABLES_DIR / "preds_HGB_test_iso.parquet",
    "xgb_val": TABLES_DIR / "preds_XGB_val_iso.parquet",
    "xgb_test": TABLES_DIR / "preds_XGB_test_iso.parquet",
}

missing_paths = [str(path) for path in required_paths.values() if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing calibrated base-model predictions:\n" + "\n".join(missing_paths))

p_lgbm_val = pd.read_parquet(required_paths["lgbm_val"])["p"].to_numpy()
p_lgbm_test = pd.read_parquet(required_paths["lgbm_test"])["p"].to_numpy()

p_hgb_val = pd.read_parquet(required_paths["hgb_val"])["p"].to_numpy()
p_hgb_test = pd.read_parquet(required_paths["hgb_test"])["p"].to_numpy()

p_xgb_val = pd.read_parquet(required_paths["xgb_val"])["p"].to_numpy()
p_xgb_test = pd.read_parquet(required_paths["xgb_test"])["p"].to_numpy()

assert len(y_val) == len(p_lgbm_val) == len(p_hgb_val) == len(p_xgb_val)
assert len(y_test) == len(p_lgbm_test) == len(p_hgb_test) == len(p_xgb_test)


def evaluate_probabilities(y_true: np.ndarray, y_prob: np.ndarray) -> dict:
    return {
        "AUROC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "Brier": brier_score_loss(y_true, y_prob),
    }


weight_grid = np.arange(0.0, 1.01, 0.05)
best_weights = None

for w_lgbm in weight_grid:
    for w_xgb in weight_grid:
        w_hgb = 1.0 - w_lgbm - w_xgb
        if w_hgb < -1e-9:
            continue

        w_hgb = max(0.0, w_hgb)

        val_prob = (
            w_lgbm * p_lgbm_val
            + w_xgb * p_xgb_val
            + w_hgb * p_hgb_val
        )

        metrics = evaluate_probabilities(y_val, val_prob)

        candidate = {
            "w_LGBM": float(w_lgbm),
            "w_XGB": float(w_xgb),
            "w_HGB": float(w_hgb),
            **metrics,
        }

        if best_weights is None:
            best_weights = candidate
        else:
            if (
                (candidate["Brier"] < best_weights["Brier"] - 1e-12)
                or (
                    abs(candidate["Brier"] - best_weights["Brier"]) <= 1e-12
                    and candidate["PR_AUC"] > best_weights["PR_AUC"] + 1e-12
                )
                or (
                    abs(candidate["Brier"] - best_weights["Brier"]) <= 1e-12
                    and abs(candidate["PR_AUC"] - best_weights["PR_AUC"]) <= 1e-12
                    and candidate["AUROC"] > best_weights["AUROC"] + 1e-12
                )
            ):
                best_weights = candidate

ensemble_val_raw = (
    best_weights["w_LGBM"] * p_lgbm_val
    + best_weights["w_XGB"] * p_xgb_val
    + best_weights["w_HGB"] * p_hgb_val
)

ensemble_test_raw = (
    best_weights["w_LGBM"] * p_lgbm_test
    + best_weights["w_XGB"] * p_xgb_test
    + best_weights["w_HGB"] * p_hgb_test
)

ensemble_raw_metrics = {
    "model": "ENS_weighted_raw",
    "split": "test",
    **evaluate_probabilities(y_test, ensemble_test_raw),
}

with open(TABLES_DIR / "ensemble_best_weights.json", "w", encoding="utf-8") as f:
    json.dump(best_weights, f, ensure_ascii=False, indent=2)

pd.DataFrame({"p": ensemble_val_raw}).to_parquet(TABLES_DIR / "preds_ENS_weighted_raw_val.parquet", index=False)
pd.DataFrame({"p": ensemble_test_raw}).to_parquet(TABLES_DIR / "preds_ENS_weighted_raw_test.parquet", index=False)

pd.DataFrame([ensemble_raw_metrics])

In [ ]:
from sklearn.isotonic import IsotonicRegression

ensemble_val_raw = pd.read_parquet(TABLES_DIR / "preds_ENS_weighted_raw_val.parquet")["p"].to_numpy()
ensemble_test_raw = pd.read_parquet(TABLES_DIR / "preds_ENS_weighted_raw_test.parquet")["p"].to_numpy()

isotonic_ensemble = IsotonicRegression(out_of_bounds="clip")
isotonic_ensemble.fit(ensemble_val_raw, y_val)

ensemble_test_iso = isotonic_ensemble.predict(ensemble_test_raw)

ensemble_metrics = pd.DataFrame([
    {
        "model": "ENS_weighted_raw",
        "split": "test",
        **evaluate_probabilities(y_test, ensemble_test_raw),
    },
    {
        "model": "ENS_weighted_iso",
        "split": "test_iso",
        **evaluate_probabilities(y_test, ensemble_test_iso),
    },
])

metrics_path = TABLES_DIR / "models_metrics.csv"

if metrics_path.exists():
    previous_metrics = pd.read_csv(metrics_path)
    previous_metrics = previous_metrics[
        ~(
            ((previous_metrics["model"] == "ENS_weighted_raw") & (previous_metrics["split"] == "test"))
            | ((previous_metrics["model"] == "ENS_weighted_iso") & (previous_metrics["split"] == "test_iso"))
        )
    ]
    ensemble_metrics = pd.concat([previous_metrics, ensemble_metrics], ignore_index=True)

ensemble_metrics.to_csv(metrics_path, index=False)

pd.DataFrame({"p": ensemble_test_iso}).to_parquet(
    TABLES_DIR / "preds_ENS_weighted_iso_test.parquet",
    index=False,
)

ensemble_metrics.tail(2)

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve, precision_recall_curve

# ROC
fpr_raw, tpr_raw, _ = roc_curve(y_test, ensemble_test_raw)
fpr_iso, tpr_iso, _ = roc_curve(y_test, ensemble_test_iso)

plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], "--", linewidth=1)
plt.plot(fpr_raw, tpr_raw, label="Ensemble RAW")
plt.plot(fpr_iso, tpr_iso, label="Ensemble Isotonic")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC — Ensemble: raw vs calibrated")
plt.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR / "roc_ensemble_raw_vs_iso.png", dpi=200)
plt.show()

# PR
precision_raw, recall_raw, _ = precision_recall_curve(y_test, ensemble_test_raw)
precision_iso, recall_iso, _ = precision_recall_curve(y_test, ensemble_test_iso)

plt.figure(figsize=(6, 5))
plt.plot(recall_raw, precision_raw, label="Ensemble RAW")
plt.plot(recall_iso, precision_iso, label="Ensemble Isotonic")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR — Ensemble: raw vs calibrated")
plt.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR / "pr_ensemble_raw_vs_iso.png", dpi=200)
plt.show()

# Calibration
frac_raw, mean_raw = calibration_curve(y_test, ensemble_test_raw, n_bins=20, strategy="quantile")
frac_iso, mean_iso = calibration_curve(y_test, ensemble_test_iso, n_bins=20, strategy="quantile")

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "--", linewidth=1)
plt.plot(mean_raw, frac_raw, "o-", label="RAW")
plt.plot(mean_iso, frac_iso, "o-", label="Isotonic")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration curve — Ensemble")
plt.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR / "calibration_ensemble_raw_vs_iso.png", dpi=200)
plt.show()

In [ ]:
metrics = pd.read_csv(TABLES_DIR / "models_metrics.csv").copy()
metrics = metrics[metrics["split"].astype(str).str.contains("test")].reset_index(drop=True)
metrics = metrics.drop_duplicates(subset=["model", "split"], keep="last")

model_order = [
    "LR",
    "LR_EN",
    "RF",
    "HGB",
    "LGBM",
    "XGB",
    "ENS_weighted_raw",
    "ENS_weighted_iso",
]

metrics["model"] = pd.Categorical(metrics["model"], categories=model_order, ordered=True)
metrics = metrics.sort_values("model").reset_index(drop=True)

metrics

In [ ]:
plot_df = metrics.copy()

plt.figure(figsize=(8, 5))
for metric_name in ["AUROC", "PR_AUC"]:
    plt.plot(plot_df["model"], plot_df[metric_name], marker="o", label=metric_name)

plt.title("Model comparison: baselines vs ensemble")
plt.xlabel("Model")
plt.ylabel("Metric value")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGS_DIR / "ensemble_vs_baselines.png", dpi=200)
plt.show()

In [ ]:
comparison_source = pd.read_csv(TABLES_DIR / "models_metrics.csv").reset_index()
latest_idx = comparison_source.groupby(["model", "split"])["index"].idxmax()

comparison_df = comparison_source.loc[latest_idx, ["model", "split", "AUROC", "PR_AUC", "Brier"]]

selected_pairs = [
    ("LR", "test_iso"),
    ("LR_EN", "test_iso"),
    ("RF", "test_iso"),
    ("HGB", "test_iso"),
    ("LGBM", "test_iso"),
    ("XGB", "test_iso"),
    ("ENS_weighted_raw", "test"),
    ("ENS_weighted_iso", "test_iso"),
]

selected_rows = []
for model_name, split_name in selected_pairs:
    row = comparison_df[(comparison_df["model"] == model_name) & (comparison_df["split"] == split_name)]
    if not row.empty:
        selected_rows.append(row.iloc[0])

final_df = pd.DataFrame(selected_rows).reset_index(drop=True)

plt.rcParams.update({"figure.dpi": 130})
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)


def plot_metric_bars(ax, metric: str, title: str, higher_is_better: bool = True) -> None:
    y_pos = np.arange(len(final_df))[::-1]
    values = final_df[metric].values[::-1]
    names = final_df["model"].values[::-1]

    bars = ax.barh(y_pos, values)

    for bar, name in zip(bars, names):
        if str(name).startswith("ENS_"):
            bar.set_hatch("//")

    pad = 0.015
    ax.set_xlim(0, np.max(values) + pad * 3)

    for y_i, value in zip(y_pos, values):
        ax.text(value + pad, y_i, f"{value:.3f}", va="center", ha="left", fontsize=10, clip_on=False)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(names, fontsize=11)
    ax.grid(axis="x", alpha=0.25, linewidth=0.8)

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_alpha(0.15)

    xlabel = f"{metric} (higher is better)" if higher_is_better else f"{metric} (lower is better)"
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_title(title, fontsize=13, pad=6, weight="bold")


plot_metric_bars(axes[0], "AUROC", "AUROC — class discrimination", higher_is_better=True)
plot_metric_bars(axes[1], "PR_AUC", "PR AUC — performance under class imbalance", higher_is_better=True)
plot_metric_bars(axes[2], "Brier", "Brier score — probabilistic calibration", higher_is_better=False)

fig.suptitle("Comparison of baseline models and the ensemble", fontsize=20, weight="bold", y=1.08)

output_path = FIGS_DIR / "comparison_triptych_final.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

# 13. Decision curve analysis

Beyond statistical accuracy, clinical usefulness must be assessed explicitly. Decision curve analysis is used to estimate whether the model provides practical benefit across different risk thresholds compared with default strategies such as treating all or treating none.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

MODELREADY_DIR = Path("./data/modelready")
TABLES_DIR = Path("./outputs/tables")
FIGS_DIR = Path("./outputs/figures")

y_test = pd.read_parquet(MODELREADY_DIR / "y_test.parquet")["y"].astype(int).to_numpy()
p_raw = pd.read_parquet(TABLES_DIR / "preds_ENS_weighted_raw_test.parquet")["p"].to_numpy()
p_iso = pd.read_parquet(TABLES_DIR / "preds_ENS_weighted_iso_test.parquet")["p"].to_numpy()


def net_benefit(y_true: np.ndarray, y_prob: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    thresholds = np.asarray(thresholds).astype(float)

    result = np.empty_like(thresholds, dtype=float)

    for i, threshold in enumerate(thresholds):
        predicted = (y_prob >= threshold).astype(int)
        tp = np.sum((predicted == 1) & (y_true == 1))
        fp = np.sum((predicted == 1) & (y_true == 0))
        result[i] = (tp / len(y_true)) - (fp / len(y_true)) * (threshold / (1 - threshold))

    return result


def treat_all_net_benefit(prevalence: float, thresholds: np.ndarray) -> np.ndarray:
    thresholds = np.asarray(thresholds).astype(float)
    odds = thresholds / (1 - thresholds)
    return prevalence - (1 - prevalence) * odds


thresholds = np.linspace(0.15, 0.55, 161)
prevalence = float(y_test.mean())

nb_raw = net_benefit(y_test, p_raw, thresholds)
nb_iso = net_benefit(y_test, p_iso, thresholds)
nb_all = treat_all_net_benefit(prevalence, thresholds)
nb_none = np.zeros_like(thresholds)

bootstrap_iterations = 400
rng = np.random.default_rng(42)

boot_iso = np.empty((bootstrap_iterations, thresholds.size), dtype=float)

for i in range(bootstrap_iterations):
    idx = rng.integers(0, len(y_test), len(y_test))
    boot_iso[i] = net_benefit(y_test[idx], p_iso[idx], thresholds)

ci_low, ci_high = np.percentile(boot_iso, [2.5, 97.5], axis=0)

plt.figure(figsize=(6.8, 5.2))
plt.plot(thresholds, nb_none, "--", linewidth=1, label="Treat none")
plt.plot(thresholds, nb_all, "--", linewidth=1, label="Treat all")
plt.plot(thresholds, nb_raw, linewidth=1.5, label="Ensemble (raw)")
plt.plot(thresholds, nb_iso, linewidth=2.3, label="Ensemble (isotonic)")
plt.fill_between(thresholds, ci_low, ci_high, alpha=0.18, linewidth=0)

plt.xlabel("Threshold probability")
plt.ylabel("Net benefit")
plt.title("Decision curve analysis — Ensemble model")
plt.xlim(0.15, 0.55)
plt.ylim(max(-0.08, np.min(ci_low)), min(0.18, np.max(ci_high) + 0.02))
plt.grid(True, linewidth=0.5, alpha=0.3)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.savefig(FIGS_DIR / "dca_ensemble_final.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix

MODELREADY_DIR = Path("./data/modelready")
TABLES_DIR = Path("./outputs/tables")
META_DIR = Path("./outputs/meta")

y_test = pd.read_parquet(MODELREADY_DIR / "y_test.parquet")["y"].astype("int8").to_numpy()
p_iso = pd.read_parquet(TABLES_DIR / "preds_ENS_weighted_iso_test.parquet")["p"].to_numpy()

n = len(y_test)
prevalence = float(y_test.mean())
threshold_grid = np.round(np.linspace(0.05, 0.70, 66), 3)


def net_benefit_at_threshold(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> float:
    predicted = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted).ravel()
    return (tp / len(y_true)) - (fp / len(y_true)) * (threshold / (1.0 - threshold))


def classification_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> dict:
    predicted = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    nb = net_benefit_at_threshold(y_true, y_prob, threshold)
    net_reduction_per_100 = (nb / (threshold / (1 - threshold))) * 100.0 if threshold < 1.0 else np.nan

    return {
        "threshold": float(threshold),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "PPV": ppv,
        "NPV": npv,
        "NetBenefit": nb,
        "NetReductionPer100": net_reduction_per_100,
    }


threshold_scan = pd.DataFrame(
    [classification_metrics(y_test, p_iso, t) for t in threshold_grid]
)

triage_zone = threshold_scan[(threshold_scan["threshold"] >= 0.20) & (threshold_scan["threshold"] <= 0.35) & (threshold_scan["NetBenefit"] >= 0)].copy()
escalation_zone = threshold_scan[(threshold_scan["threshold"] > 0.35) & (threshold_scan["threshold"] <= 0.60) & (threshold_scan["NetBenefit"] >= 0)].copy()

if not triage_zone.empty:
    triage_threshold = float(
        triage_zone.sort_values(["Sensitivity", "NetBenefit"], ascending=[False, False]).iloc[0]["threshold"]
    )
else:
    triage_threshold = float(
        threshold_scan[(threshold_scan["threshold"] >= 0.20) & (threshold_scan["threshold"] <= 0.35)]
        .sort_values("Sensitivity", ascending=False)
        .iloc[0]["threshold"]
    )

if not escalation_zone.empty:
    escalation_threshold = float(
        escalation_zone.sort_values(["PPV", "NetBenefit"], ascending=[False, False]).iloc[0]["threshold"]
    )
else:
    escalation_threshold = float(
        threshold_scan[(threshold_scan["threshold"] > 0.35) & (threshold_scan["threshold"] <= 0.60)]
        .sort_values("PPV", ascending=False)
        .iloc[0]["threshold"]
    )

triage_metrics = classification_metrics(y_test, p_iso, triage_threshold)
escalation_metrics = classification_metrics(y_test, p_iso, escalation_threshold)

threshold_scan.to_csv(TABLES_DIR / "ensemble_iso_threshold_scan_test.csv", index=False)
pd.DataFrame([
    {"policy": "triage", **triage_metrics},
    {"policy": "escalation", **escalation_metrics},
]).to_csv(TABLES_DIR / "ensemble_iso_operating_points_test.csv", index=False)

with open(META_DIR / "ensemble_iso_policy.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "prevalence": prevalence,
            "triage": triage_metrics,
            "escalation": escalation_metrics,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame([
    {"policy": "triage", **triage_metrics},
    {"policy": "escalation", **escalation_metrics},
])

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix

MODELREADY_DIR = Path("./data/modelready")
TABLES_DIR = Path("./outputs/tables")

y_test = pd.read_parquet(MODELREADY_DIR / "y_test.parquet")["y"].astype("int8").to_numpy()
p_iso = pd.read_parquet(TABLES_DIR / "preds_ENS_weighted_iso_test.parquet")["p"].to_numpy()


def net_benefit_at_threshold(y_true, y_prob, threshold):
    predicted = (y_prob >= threshold).astype("int8")
    tn, fp, fn, tp = confusion_matrix(y_true, predicted).ravel()
    weight = threshold / (1.0 - threshold)
    return (tp / len(y_true)) - (fp / len(y_true)) * weight


def threshold_metrics(y_true, y_prob, threshold):
    predicted = (y_prob >= threshold).astype("int8")
    tn, fp, fn, tp = confusion_matrix(y_true, predicted).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan
    nb = net_benefit_at_threshold(y_true, y_prob, threshold)

    return {
        "Threshold": float(threshold),
        "Sensitivity": float(sensitivity),
        "Specificity": float(specificity),
        "PPV": float(ppv),
        "Net benefit": float(nb),
    }


article_thresholds = [0.20, 0.30, 0.35]
table_numeric = pd.DataFrame([threshold_metrics(y_test, p_iso, t) for t in article_thresholds])

table_pretty = table_numeric.copy()
table_pretty["Threshold"] = table_pretty["Threshold"].map(lambda x: f"{x:.2f}")
table_pretty["Sensitivity"] = table_pretty["Sensitivity"].map(lambda x: f"{x:.3f}")
table_pretty["Specificity"] = table_pretty["Specificity"].map(lambda x: f"{x:.3f}")
table_pretty["PPV"] = table_pretty["PPV"].map(lambda x: f"{x:.3f}")
table_pretty["Net benefit"] = table_pretty["Net benefit"].map(lambda x: f"{x:.4f}")

table_numeric.to_csv(TABLES_DIR / "table2_operational_thresholds_iso_test_numeric.csv", index=False)
table_pretty.to_csv(TABLES_DIR / "table2_operational_thresholds_iso_test_pretty.csv", index=False)

table_pretty

# 14. Fairness assessment

This section evaluates whether the calibrated model behaves consistently across key demographic subgroups. The analysis focuses on potential differences in predictive performance and threshold-based behavior across sex and age categories.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

MODELREADY_DIR = Path("./data/modelready")
TABLES_DIR = Path("./outputs/tables")
META_DIR = Path("./outputs/meta")

y_test = pd.read_parquet(MODELREADY_DIR / "y_test.parquet")["y"].astype("int8").to_numpy()
p_iso = pd.read_parquet(TABLES_DIR / "preds_ENS_weighted_iso_test.parquet")["p"].to_numpy()

x_candidates = [
    MODELREADY_DIR / "X_test.parquet",
    MODELREADY_DIR / "X_test_with_meta.parquet",
    MODELREADY_DIR / "X_test_idx.parquet",
]

x_path = next((path for path in x_candidates if path.exists()), None)
if x_path is None:
    raise FileNotFoundError("Test feature file with sex/age information was not found.")

X_test = pd.read_parquet(x_path)


def infer_sex(frame: pd.DataFrame) -> pd.Series:
    cols = {col.lower(): col for col in frame.columns}

    for key in ["sex", "gender"]:
        if key in cols:
            series = frame[cols[key]]
            if series.dtype.kind in "iu":
                return series.map({1: "Female", 0: "Male"}).fillna("Unknown")
            if series.dtype.kind == "b":
                return series.map({True: "Female", False: "Male"}).fillna("Unknown")

            return (
                series.astype(str)
                .str.strip()
                .str.upper()
                .map({
                    "F": "Female",
                    "FEMALE": "Female",
                    "M": "Male",
                    "MALE": "Male",
                })
                .fillna("Unknown")
            )

    for col_name in ["sex_female", "female", "is_female"]:
        if col_name in cols:
            female = frame[cols[col_name]].astype(float).fillna(0)
            return pd.Series(np.where(female >= 0.5, "Female", "Male"), index=frame.index)

    return pd.Series(["Unknown"] * len(frame), index=frame.index)


def infer_age(frame: pd.DataFrame) -> pd.Series:
    for col_name in ["age", "age_years", "AGE", "Age"]:
        if col_name in frame.columns:
            return pd.to_numeric(frame[col_name], errors="coerce")
    return pd.Series([np.nan] * len(frame), index=frame.index)


def age_group_series(age: pd.Series) -> pd.Series:
    if age.isna().all():
        return pd.Series(["All"] * len(age), index=age.index)

    return pd.cut(
        age,
        bins=[-np.inf, 40, 65, np.inf],
        labels=["<40", "40–64", "65+"],
        right=False,
    )


sex_group = infer_sex(X_test).reset_index(drop=True)
age_group = age_group_series(infer_age(X_test)).reset_index(drop=True)

policy_path = META_DIR / "ensemble_iso_policy.json"
if policy_path.exists():
    with open(policy_path, "r", encoding="utf-8") as f:
        policy = json.load(f)
    triage_threshold = float(policy.get("triage", {}).get("threshold", 0.20))
    escalation_threshold = float(policy.get("escalation", {}).get("threshold", 0.45))
else:
    triage_threshold = 0.20
    escalation_threshold = 0.45


def group_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> dict:
    predicted = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()

    n = tn + fp + fn + tp
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan

    weight = threshold / (1 - threshold) if threshold < 0.999999 else 1e9
    net_benefit = (tp / n) - (fp / n) * weight

    return {
        "N": int(n),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "PPV": ppv,
        "NPV": npv,
        "NetBenefit": net_benefit,
    }


def evaluate_by_group(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: pd.Series,
    threshold: float,
    policy_name: str,
) -> pd.DataFrame:
    groups = groups.reset_index(drop=True)
    rows = []

    for group_name, idx in groups.groupby(groups, observed=True).groups.items():
        idx = np.fromiter(idx, dtype=int)
        metrics = group_metrics(y_true[idx], y_prob[idx], threshold)
        metrics.update({
            "policy": policy_name,
            "group": str(group_name),
            "threshold": float(threshold),
        })
        rows.append(metrics)

    return pd.DataFrame(rows)


fairness_sex = pd.concat([
    evaluate_by_group(y_test, p_iso, sex_group, triage_threshold, "TRIAGE"),
    evaluate_by_group(y_test, p_iso, sex_group, escalation_threshold, "ESCALATION"),
], ignore_index=True)

fairness_age = pd.concat([
    evaluate_by_group(y_test, p_iso, age_group, triage_threshold, "TRIAGE"),
    evaluate_by_group(y_test, p_iso, age_group, escalation_threshold, "ESCALATION"),
], ignore_index=True)


def disparity_table(frame: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    out = []
    for metric in metrics:
        pivot = frame.pivot_table(index="policy", columns="group", values=metric)
        diff = (pivot.max(axis=1) - pivot.min(axis=1)).rename(f"Δ{metric}")
        out.append(diff)
    return pd.concat(out, axis=1)


sex_disparity = disparity_table(fairness_sex, ["Sensitivity", "Specificity", "PPV", "NetBenefit"])
age_disparity = disparity_table(fairness_age, ["Sensitivity", "Specificity", "PPV", "NetBenefit"])

fairness_sex.to_csv(TABLES_DIR / "fairness_sex_ensemble_iso_test.csv", index=False)
fairness_age.to_csv(TABLES_DIR / "fairness_age_ensemble_iso_test.csv", index=False)
sex_disparity.to_csv(TABLES_DIR / "fairness_sex_disparity.csv")
age_disparity.to_csv(TABLES_DIR / "fairness_age_disparity.csv")

fairness_sex, fairness_age, sex_disparity, age_disparity

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

TABLES_DIR = Path("./outputs/tables")
FIGS_DIR = Path("./outputs/figures")

fairness_sex = pd.read_csv(TABLES_DIR / "fairness_sex_ensemble_iso_test.csv")
fairness_age = pd.read_csv(TABLES_DIR / "fairness_age_ensemble_iso_test.csv")

plot_metrics = ["Sensitivity", "PPV", "Specificity"]


def plot_fairness_panels(frame: pd.DataFrame, filename: str, x_label: str, title: str) -> None:
    policies = ["TRIAGE", "ESCALATION"]
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.8), sharey=True)

    for ax, policy_name in zip(axes, policies):
        subset = frame[frame["policy"] == policy_name].copy().sort_values("group")
        x = np.arange(len(subset))
        width = 0.25

        for i, metric in enumerate(plot_metrics):
            ax.bar(
                x + (i - (len(plot_metrics) - 1) / 2) * width,
                subset[metric].values,
                width=width,
                label=metric if policy_name == "TRIAGE" else None,
            )

        ax.set_xticks(x)
        ax.set_xticklabels(subset["group"].astype(str).tolist(), rotation=15)
        ax.set_xlabel(x_label)
        ax.set_ylim(0, 1.05)
        ax.grid(axis="y", alpha=0.25)
        ax.set_title(policy_name, fontsize=10)

    axes[0].set_ylabel("Metric value")
    axes[0].legend(loc="upper right", frameon=False, fontsize=8)

    fig.suptitle(title, fontsize=12, y=1.03)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_fairness_panels(
    fairness_sex,
    filename="fairness_sex_panels.png",
    x_label="Sex",
    title="Operational fairness by sex",
)

plot_fairness_panels(
    fairness_age,
    filename="fairness_age_panels.png",
    x_label="Age group",
    title="Operational fairness by age group",
)

# 15. Surrogate interpretation

Because the main ensemble is complex, an additional surrogate model is trained to approximate its decision logic. This provides a simpler interpretable view of the dominant factors driving predicted risk.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix

MODELREADY_DIR = Path("./data/modelready")
TABLES_DIR = Path("./outputs/tables")
FIGS_DIR = Path("./outputs/figures")
META_DIR = Path("./outputs/meta")

required_paths = {
    "y_val": MODELREADY_DIR / "y_val.parquet",
    "y_test": MODELREADY_DIR / "y_test.parquet",
    "X_val": MODELREADY_DIR / "X_val.parquet",
    "X_test": MODELREADY_DIR / "X_test.parquet",
    "preds_val_raw": TABLES_DIR / "preds_ENS_weighted_raw_val.parquet",
    "preds_test_iso": TABLES_DIR / "preds_ENS_weighted_iso_test.parquet",
}

missing_paths = [str(path) for path in required_paths.values() if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing_paths))

y_val = pd.read_parquet(required_paths["y_val"])["y"].astype("int8").to_numpy()
y_test = pd.read_parquet(required_paths["y_test"])["y"].astype("int8").to_numpy()

X_val = pd.read_parquet(required_paths["X_val"])
X_test = pd.read_parquet(required_paths["X_test"])

common_columns = [col for col in X_val.columns if col in X_test.columns]
X_val = X_val[common_columns].copy()
X_test = X_test[common_columns].copy()

p_val_raw = pd.read_parquet(required_paths["preds_val_raw"])["p"].to_numpy()
p_test_iso = pd.read_parquet(required_paths["preds_test_iso"])["p"].to_numpy()

triage_threshold = 0.20
for candidate_path in [
    META_DIR / "ensemble_iso_policy_normalized.json",
    META_DIR / "ensemble_iso_policy.json",
]:
    if candidate_path.exists():
        try:
            with open(candidate_path, "r", encoding="utf-8") as f:
                policy = json.load(f)

            if "global" in policy and "triage" in policy["global"]:
                triage_value = policy["global"]["triage"]
                triage_threshold = float(triage_value["threshold"] if isinstance(triage_value, dict) else triage_value)
            elif "triage" in policy:
                triage_value = policy["triage"]
                triage_threshold = float(triage_value["threshold"] if isinstance(triage_value, dict) else triage_value)

            break
        except Exception:
            pass

isotonic_model = IsotonicRegression(out_of_bounds="clip").fit(p_val_raw, y_val)
p_val_iso = isotonic_model.predict(p_val_raw)

y_val_decision = (p_val_iso >= triage_threshold).astype("int8")
y_test_decision = (p_test_iso >= triage_threshold).astype("int8")

surrogate_features = [
    "age_years",
    "sym_low_oxygen_saturation",
    "sym_respiratory_distress",
    "sym_dyspnea",
    "sym_fever",
    "sym_cough",
    "comorb_diabetes",
    "comorb_other_morbidity_missing",
    "sym_viral_any_missing",
]

for col in surrogate_features:
    if col not in X_val.columns:
        X_val[col] = 0.0
    if col not in X_test.columns:
        X_test[col] = 0.0

age_mean = pd.to_numeric(X_val["age_years"], errors="coerce").dropna().mean()
age_std = pd.to_numeric(X_val["age_years"], errors="coerce").dropna().std()
age_std = 1.0 if pd.isna(age_std) or age_std == 0 else age_std

X_val["age_years_z"] = (pd.to_numeric(X_val["age_years"], errors="coerce") - age_mean) / age_std
X_test["age_years_z"] = (pd.to_numeric(X_test["age_years"], errors="coerce") - age_mean) / age_std

surrogate_features = surrogate_features + ["age_years_z"]

surrogate_tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=1500,
    class_weight="balanced",
    random_state=42,
)

surrogate_tree.fit(X_val[surrogate_features], y_val_decision)

surrogate_predictions = surrogate_tree.predict(X_test[surrogate_features])

surrogate_metrics = {
    "triage_threshold": triage_threshold,
    "fidelity_accuracy": float(accuracy_score(y_test_decision, surrogate_predictions)),
    "fidelity_kappa": float(cohen_kappa_score(y_test_decision, surrogate_predictions)),
}

tn, fp, fn, tp = confusion_matrix(y_test_decision, surrogate_predictions, labels=[0, 1]).ravel()
surrogate_metrics.update({
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp),
})

pd.DataFrame([surrogate_metrics])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree, export_text

rules_text = export_text(surrogate_tree, feature_names=surrogate_features)

with open(TABLES_DIR / "surrogate_rules_triage.txt", "w", encoding="utf-8") as f:
    f.write(rules_text)

plt.figure(figsize=(16, 9))
plot_tree(
    surrogate_tree,
    feature_names=surrogate_features,
    class_names=["No triage", "Triage"],
    filled=True,
    impurity=False,
    proportion=True,
    rounded=True,
)
plt.title("Surrogate decision tree for TRIAGE policy")
plt.tight_layout()
plt.savefig(FIGS_DIR / "surrogate_tree_triage.png", dpi=220)
plt.show()

feature_importance = (
    pd.Series(surrogate_tree.feature_importances_, index=surrogate_features)
    .sort_values(ascending=False)
    .rename("importance")
    .to_frame()
)

feature_importance.to_csv(TABLES_DIR / "surrogate_feature_importance.csv")

feature_importance.head(12)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score

features_without_age = [col for col in surrogate_features if col not in ["age_years", "age_years_z"]]

surrogate_tree_no_age = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=1500,
    class_weight="balanced",
    random_state=42,
)

surrogate_tree_no_age.fit(X_val[features_without_age], y_val_decision)

predictions_no_age = surrogate_tree_no_age.predict(X_test[features_without_age])

ablation_results = pd.DataFrame([
    {
        "model": "surrogate_with_age",
        "fidelity_accuracy": float(accuracy_score(y_test_decision, surrogate_predictions)),
        "fidelity_kappa": float(cohen_kappa_score(y_test_decision, surrogate_predictions)),
    },
    {
        "model": "surrogate_without_age",
        "fidelity_accuracy": float(accuracy_score(y_test_decision, predictions_no_age)),
        "fidelity_kappa": float(cohen_kappa_score(y_test_decision, predictions_no_age)),
    },
])

ablation_results["delta_accuracy_vs_with_age"] = (
    ablation_results["fidelity_accuracy"] - ablation_results.loc[0, "fidelity_accuracy"]
)

ablation_results

# 16. TRIAGE PRO demo generation

The final stage prepares the artifacts required for a lightweight deployment-oriented demo. The goal is to show how the trained and calibrated prediction logic can be transferred into a practical decision-support prototype.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

MODELREADY_DIR = Path("./data/modelready")
TABLES_DIR = Path("./outputs/tables")
META_DIR = Path("./outputs/meta")
DEMO_DIR = Path("./demo")
DEMO_DIR.mkdir(parents=True, exist_ok=True)

y_val = pd.read_parquet(MODELREADY_DIR / "y_val.parquet")["y"].astype("int8").to_numpy()
X_val = pd.read_parquet(MODELREADY_DIR / "X_val.parquet")
X_test = pd.read_parquet(MODELREADY_DIR / "X_test.parquet")

common_columns = [col for col in X_val.columns if col in X_test.columns]
X_val = X_val[common_columns].copy()
X_test = X_test[common_columns].copy()

p_val_raw = pd.read_parquet(TABLES_DIR / "preds_ENS_weighted_raw_val.parquet")["p"].to_numpy()
isotonic_model = IsotonicRegression(out_of_bounds="clip").fit(p_val_raw, y_val)
p_val_iso = isotonic_model.predict(p_val_raw)

triage_threshold = 0.20
policy_path = META_DIR / "ensemble_iso_policy_normalized.json"
if policy_path.exists():
    with open(policy_path, "r", encoding="utf-8") as f:
        policy = json.load(f)
    triage_value = policy.get("global", {}).get("triage", 0.20)
    triage_threshold = float(triage_value.get("threshold", triage_value)) if isinstance(triage_value, dict) else float(triage_value)

features = [
    "age_years",
    "sym_low_oxygen_saturation",
    "sym_respiratory_distress",
    "sym_dyspnea",
    "sym_fever",
    "sym_cough",
    "comorb_diabetes",
    "comorb_cvd",
    "comorb_other_morbidity_missing",
    "sym_viral_any_missing",
]

for col in features:
    if col not in X_val.columns:
        X_val[col] = 0.0
    if col not in X_test.columns:
        X_test[col] = 0.0

age_series = pd.to_numeric(X_val["age_years"], errors="coerce")
age_mean = age_series.dropna().mean()
age_std = age_series.dropna().std()
age_std = 1.0 if pd.isna(age_std) or age_std == 0 else age_std

X_val["age_years_z"] = (age_series.fillna(age_mean) - age_mean) / age_std
surrogate_features = features + ["age_years_z"]

y_val_triage = (p_val_iso >= triage_threshold).astype("int8")

surrogate_tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=2000,
    class_weight="balanced",
    random_state=42,
)
surrogate_tree.fit(X_val[surrogate_features], y_val_triage)

tree_state = surrogate_tree.tree_

def export_tree_node(index: int) -> dict:
    return {
        "idx": int(index),
        "feature": None if tree_state.feature[index] < 0 else surrogate_features[tree_state.feature[index]],
        "threshold": None if tree_state.feature[index] < 0 else float(tree_state.threshold[index]),
        "left": int(tree_state.children_left[index]),
        "right": int(tree_state.children_right[index]),
        "prob1": float(tree_state.value[index][0, 1] / (tree_state.value[index][0].sum() or 1.0)),
    }

tree_json = {
    "nodes": [export_tree_node(i) for i in range(tree_state.node_count)],
    "root": 0,
    "features": surrogate_features,
    "t_triage": float(triage_threshold),
    "age_mean": float(age_mean),
    "age_std": float(age_std),
}

scaler = StandardScaler()
X_scaled = X_val[features].copy()

age_numeric = pd.to_numeric(X_val["age_years"], errors="coerce").fillna(age_mean)
X_scaled["age_years"] = scaler.fit_transform(age_numeric.to_frame())[:, 0]

logistic_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=500,
)
logistic_model.fit(X_scaled, y_val)

risk_pack = {
    "intercept": float(logistic_model.intercept_[0]),
    "coef": dict(zip(X_scaled.columns, logistic_model.coef_[0].tolist())),
    "age_mean": float(scaler.mean_[0]),
    "age_std": float(scaler.scale_[0]),
    "features": features,
}

threshold_presets = {
    "sensitive": {"label": "Sensitive", "thr": 0.25, "note": "Higher sensitivity"},
    "balanced": {"label": "Balanced", "thr": 0.35, "note": "Balanced operating point"},
    "strict": {"label": "Strict", "thr": 0.50, "note": "Higher specificity"},
}

html_source = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>TRIAGE PRO</title>
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
:root{--brand:#2563eb;--muted:#6b7280;--bg:#f3f4f6;--line:#e5e7eb;}
*{box-sizing:border-box;}
body{margin:0;background:var(--bg);font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;color:#111827;}
.shell{max-width:1100px;margin:0 auto;padding:16px 24px 32px;}
h1{margin:0 0 6px 0;font-size:26px;font-weight:700;}
small{color:#6b7280;}
.card{background:#fff;border:1px solid var(--line);border-radius:16px;padding:16px 20px;margin:12px 0;box-shadow:0 4px 12px rgba(15,23,42,.04);}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px;}
label{font-size:14px;color:#374151;}
input[type=number]{width:100%;padding:10px;border:1px solid #d1d5db;border-radius:10px;font-size:14px;}
input[type=checkbox]{margin-right:6px;}
.btn{padding:10px 14px;border:none;border-radius:10px;background:var(--brand);color:#fff;font-weight:600;cursor:pointer;font-size:14px;}
.btn.muted{background:var(--muted);}
.btn + .btn{margin-left:8px;}
.kpi{display:flex;gap:12px;flex-wrap:wrap;margin:8px 0 2px;}
.box{background:#f3f4f6;border-radius:10px;padding:10px 12px;font-size:14px;}
.badge{display:inline-block;padding:2px 8px;border-radius:999px;font-size:12px;}
.badge.ok{background:#dcfce7;color:#166534;}
.badge.no{background:#fee2e2;color:#991b1b;}
pre{background:#fafafa;border:1px dashed var(--line);padding:12px;border-radius:10px;white-space:pre-wrap;font-size:13px;max-height:260px;overflow:auto;}
.switch{display:flex;flex-wrap:wrap;gap:8px;}
.pill{padding:6px 12px;border-radius:999px;border:1px solid #d1d5db;cursor:pointer;font-size:13px;background:#fff;}
.pill.active{background:#2563eb;border-color:#1d4ed8;color:#fff;}
@media (max-width:768px){
  .shell{padding:12px 12px 24px;}
  h1{font-size:22px;}
  .card{padding:12px 14px;margin:10px 0;}
  .btn{width:100%;margin-top:8px;}
  .btn + .btn{margin-left:0;}
  .kpi{flex-direction:column;}
  pre{max-height:200px;}
}
</style>
</head>
<body>
<div class="shell">
<h1>TRIAGE PRO</h1>
<small>Offline symptom-based triage calculator</small>

<div class="card">
  <div class="switch" id="thrSwitch"></div>
  <small>TRIAGE operating threshold</small>
</div>

<div class="card">
  <div class="grid">
    <div><label>Age (years)</label><input type="number" id="age" value="60" min="0" max="120" step="1" inputmode="numeric"></div>
    <div><label><input type="checkbox" id="lowox"> Low oxygen saturation</label></div>
    <div><label><input type="checkbox" id="respdist"> Respiratory distress</label></div>
    <div><label><input type="checkbox" id="dysp"> Dyspnea</label></div>
    <div><label><input type="checkbox" id="fever"> Fever</label></div>
    <div><label><input type="checkbox" id="cough"> Cough</label></div>
    <div><label><input type="checkbox" id="dm"> Diabetes</label></div>
    <div><label><input type="checkbox" id="cvd"> Cardiovascular disease / stroke</label></div>
    <div><label><input type="checkbox" id="morbmiss"> Missing comorbidity data</label></div>
    <div><label><input type="checkbox" id="virmiss"> Missing viral symptom data</label></div>
  </div>
  <br>
  <div>
    <button class="btn" type="button" onclick="runCalc()">Calculate</button>
    <button class="btn muted" type="button" onclick="resetForm()">Reset</button>
  </div>
</div>

<div class="card">
  <h3>Result</h3>
  <div id="out"></div>
</div>

</div>

<script>
var SURR = __SURR_JSON__;
var LR = __LR_JSON__;
var PRESETS = __PRESETS_JSON__;

function sigmoid(x){return 1/(1+Math.exp(-x));}

function traverse(row){
  var nodes = SURR.nodes;
  var i = SURR.root;
  while (true){
    var n = nodes[i];
    if (n.feature === null){
      return {prob:n.prob1};
    }
    var v = row[n.feature];
    var goLeft = (v <= n.threshold);
    i = goLeft ? n.left : n.right;
  }
}

function describePatient(f) {
  var lines = [];
  lines.push("Age: " + f.age_years.toFixed(0) + " years");
  function yn(v) { return v ? "yes" : "no"; }

  lines.push("Low oxygen saturation: " + yn(f.sym_low_oxygen_saturation));
  lines.push("Respiratory distress: " + yn(f.sym_respiratory_distress));
  lines.push("Dyspnea: " + yn(f.sym_dyspnea));
  lines.push("Fever: " + yn(f.sym_fever));
  lines.push("Cough: " + yn(f.sym_cough));
  lines.push("Diabetes: " + yn(f.comorb_diabetes));
  lines.push("Cardiovascular disease / stroke: " + yn(f.comorb_cvd));
  lines.push("Missing comorbidity data: " + yn(f.comorb_other_morbidity_missing));
  lines.push("Missing viral symptom data: " + yn(f.sym_viral_any_missing));

  var txt = "";
  for (var i = 0; i < lines.length; i++){
    txt += "• " + lines[i] + "\n";
  }
  return txt;
}

function logisticRisk(f){
  var age_scaled = (f.age_years - LR.age_mean)/(LR.age_std || 1);
  var logit = LR.intercept + (LR.coef["age_years"] || 0) * age_scaled;
  var contribArr = [];
  contribArr.push("age_years (scaled): " + ((LR.coef["age_years"] || 0) * age_scaled).toFixed(4));

  for (var i = 0; i < LR.features.length; i++){
    var k = LR.features[i];
    if (k === "age_years") continue;
    var coefK = LR.coef[k] || 0;
    var val = +f[k];
    var c = coefK * val;
    logit += c;
    contribArr.push(k + ": " + c.toFixed(4));
  }

  return {p:sigmoid(logit), contrib:contribArr};
}

function collect(){
  var age = parseFloat(document.getElementById("age").value);
  if (isNaN(age)) age = 0;

  return {
    age_years: age,
    age_years_z:(age - SURR.age_mean)/(SURR.age_std || 1),
    sym_low_oxygen_saturation: document.getElementById("lowox").checked ? 1 : 0,
    sym_respiratory_distress: document.getElementById("respdist").checked ? 1 : 0,
    sym_dyspnea: document.getElementById("dysp").checked ? 1 : 0,
    sym_fever: document.getElementById("fever").checked ? 1 : 0,
    sym_cough: document.getElementById("cough").checked ? 1 : 0,
    comorb_diabetes: document.getElementById("dm").checked ? 1 : 0,
    comorb_cvd: document.getElementById("cvd").checked ? 1 : 0,
    comorb_other_morbidity_missing: document.getElementById("morbmiss").checked ? 1 : 0,
    sym_viral_any_missing: document.getElementById("virmiss").checked ? 1 : 0
  };
}

var currentThr = PRESETS["balanced"]["thr"];
var hasResult = false;

function renderSwitch(){
  var box = document.getElementById("thrSwitch");
  box.innerHTML = "";
  var order = ["sensitive","balanced","strict"];

  for (var i = 0; i < order.length; i++){
    var key = order[i];
    var p = PRESETS[key];
    if (!p) continue;
    var el = document.createElement("div");
    el.className = "pill" + (Math.abs(currentThr - p.thr) < 1e-9 ? " active" : "");
    el.textContent = p.label + " (" + p.thr.toFixed(2) + ")";
    el.title = p.note;
    el.onclick = (function(thrValue){
      return function(){
        currentThr = thrValue;
        renderSwitch();
        if (hasResult) runCalc();
      };
    })(p.thr);
    box.appendChild(el);
  }
}

function runCalc(){
  hasResult = true;
  var f = collect();
  var sur = traverse(f);
  var lr = logisticRisk(f);
  var triage = sur.prob >= currentThr;

  var patientText = describePatient(f);
  var contribText = "";
  for (var i = 0; i < lr.contrib.length; i++){
    contribText += "- " + lr.contrib[i] + "\n";
  }

  var html = ""
    + '<div class="kpi">'
    +   '<div class="box"><b>TRIAGE:</b> '
    +       (triage ? '<span class="badge ok">yes</span>' : '<span class="badge no">no</span>')
    +   '</div>'
    +   '<div class="box"><b>p(surrogate)=</b> ' + sur.prob.toFixed(4) + '</div>'
    +   '<div class="box"><b>TRIAGE threshold:</b> ' + currentThr.toFixed(2) + '</div>'
    +   '<div class="box"><b>Risk (logistic model)=</b> ' + (lr.p * 100).toFixed(1) + ' %</div>'
    + '</div>'
    + '<h4>Patient profile</h4><pre>' + patientText + '</pre>'
    + '<h4 style="margin-top:16px;">Feature contributions (logistic model)</h4><pre>' + contribText + '</pre>';

  document.getElementById("out").innerHTML = html;
}

function resetForm(){
  var boxes = document.querySelectorAll("input[type=checkbox]");
  for (var i = 0; i < boxes.length; i++){
    boxes[i].checked = false;
  }
  document.getElementById("age").value = 60;
  document.getElementById("out").innerHTML = "";
  hasResult = false;
}

renderSwitch();
</script>
</body>
</html>
"""

html = (
    html_source
    .replace("__SURR_JSON__", json.dumps(tree_json, ensure_ascii=False))
    .replace("__LR_JSON__", json.dumps(risk_pack, ensure_ascii=False))
    .replace("__PRESETS_JSON__", json.dumps(threshold_presets, ensure_ascii=False))
)

output_path = DEMO_DIR / "triage_pro_offline.html"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html)

output_path